# Primary synthetic simulation — XGBoost

**Author:** Jagruthi Nalajala  
**Paper:** *Quantifying the Revenue Degradation of Decoupled Recommendation and Promotion Systems: A Joint Causal Framework*

This notebook runs the primary synthetic experiments. It preserves the original
data-generating process, parameter grids, 60/40 held-out evaluation design,
and twenty independent seeds. The only learned-policy change is a single,
locked XGBoost regressor specification used for both the joint and decoupled
systems. The notebook reports whether it uses CUDA or CPU; this affects runtime,
not the specified model or the experiment design.

The complete potential-outcome matrix is available only because this is a
synthetic structural experiment; it is used for held-out policy evaluation,
not presented as a logged-data causal estimate.


In [1]:
# XGBoost dependency and imports
import sys
import subprocess

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost>=2.0.0', '--quiet'])
    import xgboost as xgb

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.special import softmax, expit
from xgboost import XGBRegressor

print(f'XGBoost version: {xgb.__version__}')


XGBoost version: 3.2.0


In [2]:
# Select CUDA when it is genuinely available; otherwise use CPU explicitly.
# This changes compute hardware only—not the XGBoost model or experimental design.
import json

def detect_xgb_device():
    try:
        probe = XGBRegressor(
            objective='reg:squarederror', n_estimators=1,
            tree_method='hist', device='cuda', random_state=0,
            n_jobs=1, verbosity=0,
        )
        probe.fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))
        fitted_device = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        del probe
        if fitted_device == 'cuda':
            print('XGBoost execution device: CUDA GPU')
            return 'cuda'
        print('XGBoost execution device: CPU (CUDA is not exposed to this kernel)')
    except Exception as exc:
        print(f'XGBoost execution device: CPU (CUDA probe unavailable: {exc})')
    return 'cpu'

XGB_DEVICE = detect_xgb_device()


XGBoost execution device: CPU (CUDA is not exposed to this kernel)


In [3]:
# Select sweep to run. Options: 'lambda', 'beta', 'rho'
SWEEP = 'rho'

In [4]:
# Locked learned-policy specification used in every experiment.
# Do not tune this configuration after observing the final comparison results.
def make_xgb(seed):
    return XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=10,
        subsample=0.90,
        colsample_bytree=0.90,
        tree_method='hist',
        device=XGB_DEVICE,
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )

# =============================================================
# Section 5.2: Data Generating Process Parameters
# All random seeds are fixed for reproducibility.
# =============================================================

N            = 10000                        # number of synthetic customers
D            = 10                           # customer feature dimensions
R            = 10                           # product categories
O            = 3                            # offer types: none, 10%, 25%
N_SEEDS      = 20                           # simulation seeds (Section 5.6)
DISCOUNT     = np.array([0.0, 0.10, 0.25]) # offer discount depths
SIGMA_FRAC   = 0.15                         # noise as fraction of mean baseline

# Parameter grids (Section 6)
LAMBDA_GRID  = [0.0, 0.25, 0.50, 0.75, 1.0]
BETA_GRID    = [0.2,  0.4,  0.6,  0.8,  1.0]
RHO_GRID     = [0.0, 0.25, 0.50, 0.75, 1.0]

# Baseline parameter values held fixed during each sweep
LAMBDA_BASE  = 0.75
BETA_BASE    = 0.50
RHO_BASE     = 0.50

# Fixed DGP weight matrices — RandomState(0)
# These are set once and never modified across experiments.
rng          = np.random.RandomState(0)
W_affinity   = rng.randn(R, D) * 0.5   # customer-category affinity weights
v_price      = rng.randn(D) * 0.5      # price sensitivity weights
W_baseline   = rng.randn(R, D) * 0.3   # baseline revenue weights
b_baseline   = rng.uniform(20, 80, size=R)  # baseline revenue intercepts

# Category-offer coupling matrix W — RandomState(99)
# Rows are constrained to sum to zero (Section 5.2):
# this ensures the decoupled model's marginal averaging over offers
# produces exactly zero net interaction signal, so Delta_gamma
# cannot be recovered by the decoupled system regardless of lambda.
rng2         = np.random.RandomState(99)
W_coupling   = rng2.randn(R, O)
W_coupling  -= W_coupling.mean(axis=1, keepdims=True)  # enforce zero row sums

assert np.allclose(W_coupling.sum(axis=1), 0, atol=1e-10), \
    'W_coupling row sums must be zero'
print(f'DGP initialized. Joint action space: {R} x {O} = {R*O} actions.')

DGP initialized. Joint action space: 10 x 3 = 30 actions.


In [5]:
# =============================================================
# Customer features and precomputed DGP quantities
# X ~ MVN(0, Sigma) with mild correlation structure, standardized.
# AFF, PS, MU are computed once for all N customers.
# =============================================================

def compute_affinity(X):
    """Customer-category affinity scores, normalized to sum to 1 per customer."""
    return softmax(X @ W_affinity.T, axis=1)

def compute_price_sensitivity(X):
    """Price sensitivity score in [0,1] via sigmoid transformation."""
    return expit(X @ v_price)

def compute_baseline_revenue(X):
    """Baseline revenue per customer per category. Minimum $10."""
    return np.abs(X @ W_baseline.T + b_baseline) + 10

rng_x  = np.random.RandomState(42)
A      = rng_x.randn(D, D)
Sigma  = A @ A.T / D + np.eye(D)
X      = rng_x.multivariate_normal(np.zeros(D), Sigma, size=N)
X      = (X - X.mean(axis=0)) / X.std(axis=0)

AFF    = compute_affinity(X)
PS     = compute_price_sensitivity(X)
MU     = compute_baseline_revenue(X)
SIGMA  = SIGMA_FRAC * MU.mean()

print(f'Customers: {X.shape}, SIGMA: ${SIGMA:.2f}')

Customers: (10000, 10), SIGMA: $10.08


In [6]:
# =============================================================
# Potential outcome simulators
# Implements Equation 21: Y_i(r,o) = mu(X,r) + tau(X,o) + lambda*gamma(X,r,o) + epsilon
# =============================================================

def simulate_outcomes(X_sub, AFF_sub, PS_sub, MU_sub, lambda_val, seed):
    """
    Simulate potential outcomes for lambda and beta sweeps.
    Vectorized over all (r, o) pairs.

    Parameters
    ----------
    X_sub, AFF_sub, PS_sub, MU_sub : arrays for a customer subset
    lambda_val : interaction strength (Section 4.1)
    seed : random seed for noise draws

    Returns
    -------
    Y : (N_sub, R, O) potential outcome array
    """
    np.random.seed(seed)
    N_sub   = len(X_sub)
    Y       = np.zeros((N_sub, R, O))
    mu_mean = MU_sub.mean(axis=1)  # per-customer mean baseline across categories
    for r in range(R):
        for o in range(O):
            base  = MU_sub[:, r] + PS_sub * DISCOUNT[o] * mu_mean
            gamma = AFF_sub[:, r] * MU_sub[:, r] * W_coupling[r, o] * PS_sub
            eps   = np.random.normal(0, SIGMA, N_sub)
            Y[:, r, o] = np.maximum(base + lambda_val * gamma + eps, 0)
    return Y


def simulate_outcomes_rho(X_sub, AFF_sub, PS_sub, MU_sub, lambda_val, rho, seed):
    """
    Simulate potential outcomes for the rho sweep.
    Includes recommendation-induced confounding term that varies with rho
    (Section 4.2). The confounding term conf = rho*(rec_weight[r] - 1/R)*MU
    produces the non-monotonic relationship between rho and Delta:
    at rho=0 confounding is zero, at rho=0.25 it peaks, at rho=1.0
    the recommendation distribution collapses and confounding disappears.

    Parameters
    ----------
    rho : recommendation personalization level in [0, 1] (Section 4.2)
    """
    np.random.seed(seed)
    N_sub   = len(X_sub)
    Y       = np.zeros((N_sub, R, O))
    mu_mean = MU_sub.mean(axis=1)
    for i in range(N_sub):
        if rho == 0.0:
            rec_weight = np.ones(R) / R
        elif rho == 1.0:
            rec_weight = np.zeros(R)
            rec_weight[AFF_sub[i].argmax()] = 1.0
        else:
            alpha      = (1.0 - rho) * R * AFF_sub[i] + 1e-6
            rec_weight = np.random.dirichlet(alpha)
        for r in range(R):
            for o in range(O):
                base  = MU_sub[i, r] + PS_sub[i] * DISCOUNT[o] * mu_mean[i]
                conf  = rho * (rec_weight[r] - 1.0 / R) * MU_sub[i, r]
                gamma = AFF_sub[i, r] * MU_sub[i, r] * W_coupling[r, o] * PS_sub[i]
                eps   = np.random.normal(0, SIGMA)
                Y[i, r, o] = np.maximum(base + lambda_val * gamma + conf + eps, 0)
    return Y

print('Simulators defined.')

Simulators defined.


In [7]:
# =============================================================
# Section 5.5: Joint and decoupled system implementations
# Section 5.6: Evaluation metrics (IRC, PEHE, oracle gap)
# =============================================================

def run_experiment(seed, lam, rho, beta):
    """
    Run one seed of the joint vs. decoupled comparison.

    Design follows the corrected bootstrap procedure:
    - 60/40 train/test split on customer indices
    - Train and test outcomes generated with independent random seeds
      (seed*100 for train, seed*100+1 for test) to prevent data leakage
    - Models trained on train customers, evaluated on held-out test customers
    - Budget constraint applied via greedy allocation (Section 4.3)

    Returns dict with oracle gap, estimated gap, and PEHE.
    """
    rng_s  = np.random.RandomState(seed * 13)
    idx    = rng_s.permutation(N)
    n_tr   = int(0.6 * N)
    tr_idx = idx[:n_tr]
    te_idx = idx[n_tr:]
    n_te   = len(te_idx)

    X_tr,  X_te   = X[tr_idx],   X[te_idx]
    AFF_tr, AFF_te = AFF[tr_idx], AFF[te_idx]
    PS_tr,  PS_te  = PS[tr_idx],  PS[te_idx]
    MU_tr,  MU_te  = MU[tr_idx],  MU[te_idx]

    # Generate outcomes with separate seeds for train and test
    sim_fn = simulate_outcomes_rho if SWEEP == 'rho' else simulate_outcomes
    if SWEEP == 'rho':
        Y_tr = sim_fn(X_tr, AFF_tr, PS_tr, MU_tr, lam, rho, seed * 100)
        Y_te = sim_fn(X_te, AFF_te, PS_te, MU_te, lam, rho, seed * 100 + 1)
    else:
        Y_tr = sim_fn(X_tr, AFF_tr, PS_tr, MU_tr, lam, seed * 100)
        Y_te = sim_fn(X_te, AFF_te, PS_te, MU_te, lam, seed * 100 + 1)

    base_tr    = Y_tr[:, 0, 0]
    incr_tr    = Y_tr - base_tr[:, None, None]
    flat_tr    = incr_tr.reshape(n_tr, -1)
    tau_rec_tr = incr_tr.mean(axis=2)
    tau_off_tr = incr_tr.mean(axis=1)

    base_te    = Y_te[:, 0, 0]
    incr_te    = Y_te - base_te[:, None, None]
    flat_te    = incr_te.reshape(n_te, -1)
    tau_rec_te = incr_te.mean(axis=2)
    tau_off_te = incr_te.mean(axis=1)
    k          = max(1, int(beta * n_te))

    # Oracle gap: uses true potential outcomes, no model training
    best_j_act = flat_te.argmax(axis=1)
    top_j      = np.argsort(flat_te.max(axis=1))[::-1][:k]
    mask_j     = np.zeros(n_te, bool); mask_j[top_j] = True
    oracle_j   = flat_te[np.arange(n_te), best_j_act][mask_j].mean()

    br_or  = tau_rec_te.argmax(axis=1)
    bo_or  = tau_off_te.argmax(axis=1)
    top_d  = np.argsort(tau_rec_te.max(axis=1) + tau_off_te.max(axis=1))[::-1][:k]
    mask_d = np.zeros(n_te, bool); mask_d[top_d] = True
    oracle_d     = incr_te[np.arange(n_te), br_or, bo_or][mask_d].mean()
    oracle_delta = oracle_j - oracle_d
    oracle_pct   = oracle_delta / oracle_j * 100 if oracle_j > 0 else 0.0

    # Joint system: one XGBoost model per (r,o) action — 30 models total
    joint_m = [make_xgb(seed + a).fit(X_tr, flat_tr[:, a])
               for a in range(R * O)]

    # Decoupled system: 10 product XGBoost models + 3 offer XGBoost models
    rec_m = [make_xgb(seed + r).fit(X_tr, tau_rec_tr[:, r])
               for r in range(R)]
    off_m = [make_xgb(seed + 100 + o).fit(X_tr, tau_off_tr[:, o])
               for o in range(O)]

    jh = np.column_stack([m.predict(X_te) for m in joint_m])
    rh = np.column_stack([m.predict(X_te) for m in rec_m])
    oh = np.column_stack([m.predict(X_te) for m in off_m])

    # IRC with budget constraint (greedy allocation, Section 4.3)
    baj  = jh.argmax(axis=1)
    topj = np.argsort(jh.max(axis=1))[::-1][:k]
    mj   = np.zeros(n_te, bool); mj[topj] = True
    est_j = np.where(mj, [flat_te[i, baj[i]] for i in range(n_te)], 0.0).mean()

    brd  = rh.argmax(axis=1)
    bod  = oh.argmax(axis=1)
    topd = np.argsort(rh.max(axis=1) + oh.max(axis=1))[::-1][:k]
    md   = np.zeros(n_te, bool); md[topd] = True
    est_d = np.where(md, [incr_te[i, brd[i], bod[i]] for i in range(n_te)], 0.0).mean()

    est_delta = est_j - est_d
    est_pct   = est_delta / est_j * 100 if est_j > 0 else 0.0

    # PEHE (Equation 23)
    pehe_j = np.sqrt(((jh - flat_te) ** 2).mean())
    pehe_d = np.sqrt((((rh - tau_rec_te)**2).mean() +
                      ((oh - tau_off_te)**2).mean()) / 2)

    return dict(
        oracle_joint=oracle_j,   oracle_dec=oracle_d,
        oracle_delta=oracle_delta, oracle_pct=oracle_pct,
        est_joint=est_j,         est_dec=est_d,
        est_delta=est_delta,     est_pct=est_pct,
        pehe_joint=pehe_j,       pehe_dec=pehe_d
    )

print('Experiment function defined.')

Experiment function defined.


In [8]:
# =============================================================
# Run selected sweep
# Results saved incrementally after each parameter value.
# =============================================================

sweep_config = {
    'lambda': (LAMBDA_GRID, 'lambda', LAMBDA_BASE, RHO_BASE,  BETA_BASE),
    'beta':   (BETA_GRID,   'beta',   LAMBDA_BASE, RHO_BASE,  BETA_BASE),
    'rho':    (RHO_GRID,    'rho',    LAMBDA_BASE, RHO_BASE,  BETA_BASE),
}

param_grid, param_name, lam_base, rho_base, beta_base = sweep_config[SWEEP]
all_rows = []

print(f'Running {param_name} sweep: {len(param_grid)} values x {N_SEEDS} seeds')
if SWEEP != 'rho':
    print(f'Fixed: lambda={lam_base}, rho={rho_base}, beta={beta_base}')
else:
    print(f'Fixed: lambda={lam_base}, beta={beta_base}')
print()

for param_val in param_grid:
    lam  = param_val if SWEEP == 'lambda' else lam_base
    beta = param_val if SWEEP == 'beta'   else beta_base
    rho  = param_val if SWEEP == 'rho'    else rho_base

    seed_rows = []
    for seed in range(N_SEEDS):
        res = run_experiment(seed=seed, lam=lam, rho=rho, beta=beta)
        res[param_name] = param_val
        res['seed']     = seed
        seed_rows.append(res)

    all_rows.extend(seed_rows)
    df_p = pd.DataFrame(seed_rows)
    print(f'{param_name}={param_val:.2f} | '
          f'Oracle: ${df_p.oracle_delta.mean():.2f} ({df_p.oracle_pct.mean():.1f}%) | '
          f'Estimated: ${df_p.est_delta.mean():.2f} ± {df_p.est_delta.std():.2f} '
          f'({df_p.est_pct.mean():.1f}%)')

    pd.DataFrame(all_rows).to_csv(f'results_{SWEEP}_partial.csv', index=False)

df_results = pd.DataFrame(all_rows)
df_results.to_csv(f'results_{SWEEP}.csv', index=False)
print(f'\nSaved: results_{SWEEP}.csv ({len(df_results)} rows)')

Running rho sweep: 5 values x 20 seeds
Fixed: lambda=0.75, beta=0.5

rho=0.00 | Oracle: $8.71 (11.4%) | Estimated: $0.94 ± 0.10 (3.2%)
rho=0.25 | Oracle: $9.32 (12.0%) | Estimated: $2.68 ± 0.13 (8.7%)
rho=0.50 | Oracle: $8.77 (10.5%) | Estimated: $2.77 ± 0.20 (8.4%)
rho=0.75 | Oracle: $7.81 (8.2%) | Estimated: $2.55 ± 0.21 (7.0%)
rho=1.00 | Oracle: $7.80 (5.8%) | Estimated: $1.85 ± 0.33 (3.5%)

Saved: results_rho.csv (100 rows)


In [9]:
# =============================================================
# Summary table matching paper format
# =============================================================

print(f'Table: {param_name.capitalize()} sweep results')

header = '{:<8} {:>10} {:>9} {:>8} {:>8} {:>7} {:>8} {:>8}'.format(
    param_name, 'Oracle Δ', 'Oracle%', 'Est Δ', 'Est SD', 'Est%', 'PEHE_j', 'PEHE_d'
)
print(header)
print('-' * 72)

for pv in param_grid:
    d = df_results[df_results[param_name] == pv]
    print(
        f'{pv:<8.2f} '
        f'${d.oracle_delta.mean():>8.2f} '
        f'{d.oracle_pct.mean():>8.1f}% '
        f'${d.est_delta.mean():>6.2f} '
        f'${d.est_delta.std():>6.2f} '
        f'{d.est_pct.mean():>6.1f}% '
        f'{d.pehe_joint.mean():>8.2f} '
        f'{d.pehe_dec.mean():>8.2f}'
    )

Table: Rho sweep results
rho        Oracle Δ   Oracle%    Est Δ   Est SD    Est%   PEHE_j   PEHE_d
------------------------------------------------------------------------
0.00     $    8.71     11.4% $  0.94 $  0.10    3.2%    14.40    11.15
0.25     $    9.32     12.0% $  2.68 $  0.13    8.7%    14.50    11.20
0.50     $    8.77     10.5% $  2.77 $  0.20    8.4%    15.21    11.78
0.75     $    7.81      8.2% $  2.55 $  0.21    7.0%    17.29    13.56
1.00     $    7.80      5.8% $  1.85 $  0.33    3.5%    23.14    18.57
